In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
    substring,
    trim,
    when,
    abs
)

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()


In [0]:
def load_curated_payment_schedule_profits(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :
        date_filter = f"businessdate = {run_date!r}"

        # schema = (
        #     "PaymentScheduleId string, SourceSystem string, PortfolioKey string, PortfolioName string,"
        #     " IngestionTimestamp timestamp,  BatchId string, PaymentScheduleKey string,"
        #     "ExposureKey String, PartyKey string, AgreementKey string, SettlementKey String, PaymentScheduleTypeCode string, PaymentScheduleType string,"
        #     "InstallmentNumber int, InstallmentDate timestamp, RequestSN int, InstallmentTypeCode string,"
        #     "InstallmentType string, InstallmentStatusCode string, InstallmentStatus string,"
        #     "InstallmentAmount decimal(38,7), InterestAmount decimal(19,2), CapitalAmount decimal(19,2),"
        #     "CommissionAmount decimal(19,2), ExpenseAmount decimal(19,2),"
        #     "OverdueDays int, InstallmentAmountPaid decimal(19,2), "
        #     "InterestAmountPaid decimal(19,2), CapitalAmountPaid decimal(19,2),"
        #     "CommissionAmountPaid decimal(19,2), ExpenseAmountPaid decimal(19,2),"
        #     "RemainingAmount decimal(19,2), DownPaymentFlag boolean, RecordKey int,"
        #     "SettlementCaseRecordKey int, PayTolerance int, PayPercent decimal(12,6), PayPercentAmount decimal(18,2),"
        #     "ImplementationFlag boolean, BusinessDate date"
        # )

        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.PaymentSchedule"

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                spark,
                curated_table_path,
                run_date,  # type: ignore
                "profits",
            )        
        else:
            logging.info("Tables does not exist, no need to delete entries for business date!")

        # ================== Portfolio ==================

        portfolio_extra_filters = None

        if portfolio_extra_filters is not None:
            portfolio_extra_filters.append(date_filter)
            portfolio_final_filters = " AND ".join(portfolio_extra_filters)
        else:
            portfolio_final_filters = date_filter

        portfolio_selected_fields = ["PortfolioName", "unit_code_profits"]

        portfolio = load_landed_portfolio_table(
            spark,
            curated_zone,  # type: ignore
            "portfolio",
            portfolio_final_filters,
            portfolio_selected_fields,
        )
        # ================== payment_plan ==================

        payment_plan_extra_filters = None

        if payment_plan_extra_filters is not None:
            payment_plan_extra_filters.append(date_filter)
            payment_plan_final_filters = " AND ".join(payment_plan_extra_filters)
        else:
            payment_plan_final_filters = date_filter

        payment_plan_selected_fields = [
            "ACCOUNT_NUMBER",
            "REQUEST_TYPE",
            "REQUEST_DT",
            "UNIT_CODE",
            "INSTALL_SN",
            "REQUEST_SN",
            "REQUEST_STATUS",
            "INSTALL_AMN",
            "INTEREST_AMN",
            "CAPITAL_AMN",
            "COMMISSION_AMN",
            "EXPENSE_AMN",
            "rq_overdue_days",
            "rq_cap_paid",
            "rq_exp_paid",
            "rq_com_paid",
            "rq_int_paid",
            "rq_remaining_amnt",
            "businessdate",
        ]

        payment_plan = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "profits_payment_plan",
            payment_plan_final_filters,
            payment_plan_selected_fields,
        )

        # Get distinct values based on selected fields and new column
        payment_plan = payment_plan.distinct()

        payment_plan = payment_plan.withColumn(
            "Account_Number_Without_Prefix",
            substring(payment_plan["Account_Number"], 4, 50),
        )

        # ================== clc_pay_promises ==================

        clc_pay_promises_extra_filters = None

        if clc_pay_promises_extra_filters is not None:
            clc_pay_promises_extra_filters.append(date_filter)
            clc_pay_promises_final_filters = " AND ".join(clc_pay_promises_extra_filters)
        else:
            clc_pay_promises_final_filters = date_filter

        clc_pay_promises_selected_fields = [
            "CASE_ID",
            "CUST_ID",
            "PAY_NUMBER",
            "PAY_DT",
            "PAY_AMN",
            "CREATE_UNIT",
            "PAY_COMMENTS",
            "PAID_AMN",
            "PAID_DT",
            "PAY_RESULT_STS",
            "PAY_PERCENT",
            "PAY_PERCENT_AMN",
            "PAY_TOLERANCE",
            "PAY_PROMISE_ID",
            "BusinessDate",
        ]

        clc_pay_promises = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "profits_clc_pay_promises",
            clc_pay_promises_final_filters,
            clc_pay_promises_selected_fields,
        )

        # ================== clc_case ==================

        clc_case_extra_filters = None

        if clc_case_extra_filters is not None:
            clc_case_filters.append(date_filter)
            clc_case_filters = " AND ".join(clc_case_extra_filters)
        else:
            clc_case_final_filters = date_filter

        clc_case_selected_fields = [
            "CASE_ACC_NUMBER",
            "CASE_ACC_SYSTEM",
            "CASE_ID",
            "CASE_CUSTOMER_ID",
            "BusinessDate",
            "CASE_UNIT",
        ]

        clc_case = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "profits_clc_case",
            clc_case_final_filters,
            clc_case_selected_fields,
        )

        # Joins Condition
        payment_plan_portfolio_join_condition = (
            payment_plan["UNIT_CODE"] == portfolio["unit_code_profits"]
        )

        clc_case_clc_pay_promises_join_condition = (
            clc_case["CASE_ID"] == clc_pay_promises["CASE_ID"]
        )

        clc_case_portfolio_join_condition = (
            clc_case["CASE_UNIT"] == portfolio["unit_code_profits"]
        )

        # Joins
        target_table_profitsA = payment_plan.join(
            portfolio, payment_plan_portfolio_join_condition, "left"
        )
        
        target_table_profitsB = clc_pay_promises.join(
            clc_case, clc_case_clc_pay_promises_join_condition, "left"
        ).join(
            portfolio, clc_case_portfolio_join_condition, "left"
        )

        # HashKeys
        target_table_profitsA_PaymentScheduleId_primary_key_fields = [
            payment_plan.ACCOUNT_NUMBER,
            payment_plan.REQUEST_TYPE,
            payment_plan.REQUEST_DT,
            payment_plan.REQUEST_SN,
            payment_plan.INSTALL_SN,
            lit("profits"),
        ]

        target_table_profitsA = (
            target_table_profitsA.withColumn(
                "PaymentScheduleId",
                sha2(
                    concat_ws(
                        ";", *target_table_profitsA_PaymentScheduleId_primary_key_fields
                    ),
                    256,
                ),
            )
            .withColumn(
                "PaymentScheduleKey",
                concat_ws(
                    ";",
                    payment_plan.ACCOUNT_NUMBER,
                    payment_plan.REQUEST_TYPE,
                    payment_plan.REQUEST_DT,
                    payment_plan.REQUEST_SN,
                    payment_plan.INSTALL_SN,
                ),
            )
            .withColumn(
                "InstallmentAmountPaid",
                (
                    payment_plan.rq_cap_paid
                    + payment_plan.rq_exp_paid
                    + payment_plan.rq_com_paid
                    + payment_plan.rq_int_paid
                ),
            )
            .withColumn(
                "InstallmentStatus",
                when(payment_plan.REQUEST_STATUS == "", lit("Active"))
                .when(payment_plan.REQUEST_STATUS == "1", lit("Active"))
                .when(payment_plan.REQUEST_STATUS == "2", lit("Paid"))
                .otherwise(lit("Overdue")),
            ).withColumn(
                "InstallmentType",
                when(payment_plan.REQUEST_TYPE == "1", lit("Installment"))
                .when(payment_plan.REQUEST_TYPE == "2", lit("Interest Request"))
                .when(payment_plan.REQUEST_TYPE == "3", lit("Loan Request"))
                .when(payment_plan.REQUEST_TYPE == "4", lit("ExpenseRequest"))
                .when(payment_plan.REQUEST_TYPE == "5", lit("CapitalRequest"))
            )
        )

        # HashKeys
        target_table_profitsB_PaymentScheduleId_primary_key_fields = [
            clc_pay_promises.PAY_PROMISE_ID,
            clc_pay_promises.CUST_ID,
            clc_pay_promises.PAY_NUMBER,
            lit("profits"),
        ]

        target_table_profitsB = (
            target_table_profitsB.withColumn(
                "PaymentScheduleId",
                sha2(
                    concat_ws(
                        ";", *target_table_profitsB_PaymentScheduleId_primary_key_fields
                    ),
                    256,
                ),
            ).withColumn(
                "PaymentScheduleKey",
                concat_ws(
                    ";",
                    clc_pay_promises.PAY_PROMISE_ID,
                    clc_pay_promises.CUST_ID,
                    clc_pay_promises.PAY_NUMBER,
                ),
            ).withColumn(
                "AgreementKey",
                when(
                    col("CASE_ACC_SYSTEM") == 19, 
                    col("CASE_ACC_NUMBER")
                ).otherwise(None)
            ).withColumn(
                "ExposureKey",
                when(
                    col("CASE_ACC_SYSTEM") == 4,
                    col("CASE_ACC_NUMBER")
                ).otherwise(None)
            ).withColumn(
                "DownPaymentFlag",
                (
                    when(
                        (col("PAY_PERCENT_AMN") != 0) | (col("PAY_PERCENT_AMN").isNotNull()),
                        True
                    ).otherwise(False)
                ),
            ).withColumn(
                "InstallmentStatus",
                when(col("PAY_RESULT_STS")  == 1, "ΜΕΛΛΟΝΤΙΚΗ")
                .when(col("PAY_RESULT_STS") == 2, "ΚΑΘΥΣΤΕΡΗΜΕΝΗ")
                .when(col("PAY_RESULT_STS") == 3, "ΠΛΗΡΩΜΕΝΗ ΟΛΙΚΩΣ")
                .when(col("PAY_RESULT_STS") == 4, "ΠΛΗΡΩΜΕΝΗ ΜΕΡΙΚΩΣ")
                .otherwise(None)
            )
        )

        # ================== Selection of Final Columns for profitsA ==================

        result_df_profitsA = target_table_profitsA.select(
            target_table_profitsA.PaymentScheduleId.alias("PaymentScheduleId"),
            lit("profits").cast("String").alias("SourceSystem"),
            portfolio.unit_code_profits.cast("String").alias("PortfolioKey"),
            portfolio.PortfolioName.cast("String").alias("PortfolioName"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens"))
            .cast("timestamp")
            .alias("IngestionTimestamp"),
            lit(None).cast("String").alias("BatchId"),
            target_table_profitsA.PaymentScheduleKey.cast("String").alias(
                "PaymentScheduleKey"
            ),
            payment_plan.ACCOUNT_NUMBER.cast("String").alias("ExposureKey"),
            lit(None).cast("String").alias("PartyKey"),
            lit(None).cast("String").alias("AgreementKey"),
            lit(None).cast("String").alias("SettlementKey"),
            lit("Loan Payment Schedule")
            .cast("String")
            .alias("PaymentScheduleTypeCode"),
            lit("Loan Payment Schedule").cast("String").alias("PaymentScheduleType"),
            payment_plan.INSTALL_SN.cast("Int").alias("InstallmentNumber"),
            payment_plan.REQUEST_DT.cast("timestamp").alias("InstallmentDate"),
            payment_plan.REQUEST_SN.cast("Int").alias("RequestSN"),
            payment_plan.REQUEST_TYPE.cast("String").alias("InstallmentTypeCode"),
            target_table_profitsA.InstallmentType.cast("String").alias("InstallmentType"),
            payment_plan.REQUEST_STATUS.cast("String").alias("InstallmentStatusCode"),
            target_table_profitsA.InstallmentStatus.cast("String").alias(
                "InstallmentStatus"
            ),

            (payment_plan.INSTALL_AMN.cast("decimal(38,7)") * -1).cast("decimal(38,7)").alias("InstallmentAmount"),
            (payment_plan.INTEREST_AMN.cast("decimal(19,2)") * -1).cast("decimal(19,2)").alias("InterestAmount"),
            (payment_plan.CAPITAL_AMN.cast("decimal(19,2)") * -1).cast("decimal(19,2)").alias("CapitalAmount"),

            payment_plan.COMMISSION_AMN.cast("decimal(19,2)").alias("CommissionAmount"),
            abs(payment_plan.EXPENSE_AMN.cast("decimal(19,2)")).alias("ExpenseAmount"),
            payment_plan.rq_overdue_days.cast("Int").alias("OverdueDays"),
            target_table_profitsA.InstallmentAmountPaid.cast("decimal(19,2)").alias(
                "InstallmentAmountPaid"
            ),
            payment_plan.rq_int_paid.cast("decimal(19,2)").alias("InterestAmountPaid"),
            payment_plan.rq_cap_paid.cast("decimal(19,2)").alias("CapitalAmountPaid"),
            payment_plan.rq_com_paid.cast("decimal(19,2)").alias(
                "CommissionAmountPaid"
            ),
            payment_plan.rq_exp_paid.cast("decimal(19,2)").alias("ExpenseAmountPaid"),
            abs(payment_plan.rq_remaining_amnt.cast("decimal(19,2)")).alias(
                "RemainingAmount"
            ),
            lit(None).cast("boolean").alias("DownPaymentFlag"),
            lit(None).cast("int").alias("RecordKey"),
            lit(None).cast("int").alias("SettlementCaseRecordKey"),
            lit(None).cast("int").alias("PayTolerance"),
            lit(None).cast("decimal(12,6)").alias("PayPercent"),
            lit(None).cast("decimal(18,2)").alias("PayPercentAmount"),
            lit(None).cast("boolean").alias("ImplementationFlag"),
            payment_plan.businessdate.cast("date").alias("BusinessDate"),
        )

        # ================== Selection of Final Columns for profitsB table ==================

        result_df_profitsB = target_table_profitsB.select(
            target_table_profitsB.PaymentScheduleId.alias("PaymentScheduleId"),
            lit("profits").cast("String").alias("SourceSystem"),
            portfolio.unit_code_profits.cast("String").alias("PortfolioKey"),
            portfolio.PortfolioName.cast("String").alias("PortfolioName"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens"))
            .cast("timestamp")
            .alias("IngestionTimestamp"),
            lit(None).cast("String").alias("BatchId"),
            target_table_profitsB.PaymentScheduleKey.cast("String").alias(
                "PaymentScheduleKey"
            ),
            trim(target_table_profitsB.ExposureKey).cast("String").alias("ExposureKey"),
            target_table_profitsB.CUST_ID.cast("String").alias("PartyKey"),
            trim(target_table_profitsB.AgreementKey).cast("String").alias("AgreementKey"),
            target_table_profitsB.PAY_PROMISE_ID.cast("String").alias("SettlementKey"),
            lit(None).cast("String").alias("PaymentScheduleTypeCode"),
            lit(None).cast("String").alias("PaymentScheduleType"),
            target_table_profitsB.PAY_NUMBER.cast("Int").alias("InstallmentNumber"),
            target_table_profitsB.PAY_DT.cast("timestamp").alias("InstallmentDate"),
            lit(None).cast("Int").alias("RequestSN"),
            lit(None).cast("String").alias("InstallmentTypeCode"),
            lit(None).cast("String").alias("InstallmentType"),
            target_table_profitsB.PAY_RESULT_STS.cast("String").alias("InstallmentStatusCode"),
            target_table_profitsB.InstallmentStatus.cast("String").alias(
                "InstallmentStatus"
            ),
            abs(target_table_profitsB.PAY_AMN.cast("decimal(38,7)")).alias("InstallmentAmount"),
            lit(None).cast("decimal(19,2)").alias("InterestAmount"),
            lit(None).cast("decimal(19,2)").alias("CapitalAmount"),
            lit(None).cast("decimal(19,2)").alias("CommissionAmount"),
            lit(None).cast("decimal(19,2)").alias("ExpenseAmount"),
            lit(None).cast("Int").alias("OverdueDays"),
            target_table_profitsB.PAID_AMN.cast("decimal(19,2)").alias(
                "InstallmentAmountPaid"
            ),
            lit(None).cast("decimal(19,2)").alias("InterestAmountPaid"),
            lit(None).cast("decimal(19,2)").alias("CapitalAmountPaid"),
            lit(None).cast("decimal(19,2)").alias(
                "CommissionAmountPaid"
            ),
            lit(None).cast("decimal(19,2)").alias("ExpenseAmountPaid"),
            lit(None).cast("decimal(19,2)").alias(
                "RemainingAmount"
            ),
            target_table_profitsB.DownPaymentFlag.cast("boolean").alias("DownPaymentFlag"),
            lit(None).cast("int").alias("RecordKey"),
            lit(None).cast("int").alias("SettlementCaseRecordKey"),
            target_table_profitsB.PAY_TOLERANCE.cast("int").alias("PayTolerance"),
            target_table_profitsB.PAY_PERCENT.cast("decimal(12,6)").alias("PayPercent"),
            target_table_profitsB.PAY_PERCENT_AMN.cast("decimal(18,2)").alias("PayPercentAmount"),
            lit(None).cast("boolean").alias("ImplementationFlag"),
            clc_pay_promises.BusinessDate.cast("date").alias("BusinessDate"),
        )

        result_union_df = result_df_profitsA.union(result_df_profitsB)

        result_union_df.write.format("delta").clusterBy("BusinessDate").mode(
            "append"
        ).saveAsTable(curated_table_path)

    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")


In [0]:
load_curated_payment_schedule_profits(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix)